In [15]:
import pandas as pd
import numpy as np
import re
import yfinance as yf

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("deep")

from langdetect import detect

print("Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς!")

Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς!


In [16]:
df = pd.read_csv("../data/CBS_dataset_v1.0.csv")

print("=" * 50)
print("ΒΑΣΙΚΕΣ ΠΛΗΡΟΦΟΡΙΕΣ DATASET")
print("=" * 50)
print(f"Συνολικές ομιλίες   : {len(df):,}")
print(f"Στήλες              : {list(df.columns)}")
print(f"Χρονικό εύρος       : {df['Date'].min()} → {df['Date'].max()}")
print(f"Χώρες               : {df['Country'].nunique()}")
print(f"Κεντρικές Τράπεζες  : {df['CentralBank'].nunique()}")
print(df.isnull().sum())

ΒΑΣΙΚΕΣ ΠΛΗΡΟΦΟΡΙΕΣ DATASET
Συνολικές ομιλίες   : 35,487
Στήλες              : ['URL', 'PDF', 'Title', 'Subtitle', 'Date', 'Authorname', 'Role', 'Gender', 'CentralBank', 'Country', 'text', 'text_original', 'Filename', 'Language', 'Source']
Χρονικό εύρος       : 1986-01-06 → 2023-12-29
Χώρες               : 131
Κεντρικές Τράπεζες  : 143
URL               1652
PDF              11019
Title                0
Subtitle         13851
Date                 0
Authorname           0
Role                 0
Gender               0
CentralBank          0
Country              0
text                 0
text_original    30140
Filename             0
Language             0
Source               0
dtype: int64


In [17]:
df.head()

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source
0,https://www.cbaruba.org/readBlob.do?id=10756,NaN,President speech Managing the Economy as if th...,NaN,2021-12-08,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Managing the Economy as if the Future Really M...,NaN,abw_10756,English,CB websites
1,https://www.cbaruba.org/readBlob.do?id=7515,NaN,Speech President of the CBA 4th Annual Future ...,NaN,2019-11-01,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Safeguarding our Future: Strategies for an Aru...,NaN,abw_7515,English,CB websites
2,https://www.cbaruba.org/readBlob.do?id=7518,NaN,Speech Symposium President Semeleer CBA,NaN,2019-09-06,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,FOSTERING ECONOMIC RESILIENCE IN ARUBA; FROM R...,NaN,abw_7518,English,CB websites
3,https://www.cbaruba.org/readBlob.do?id=7548,NaN,Integrity Koninkrijk Seminar,NaN,2016-10-28,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,"Presentation by Mrs Jeanette R. Semeleer, Pres...","Voordracht door mevrouw Jeanette R. Semeleer, ...",abw_7548,Dutch,CB websites
4,https://www.cbaruba.org/readBlob.do?id=7554,NaN,Speech by the President at the BES seminar hel...,NaN,2010-03-29,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Ongoing changes in the supervisory landscape a...,NaN,abw_7554,English,CB websites


In [18]:
cols = ['Date', 'text', 'CentralBank', 'Country', 'Authorname', 'Language', 'Role']
df = df[cols]

In [19]:
def detect_language(text):
    """Εντοπίζει τη γλώσσα του κειμένου."""
    if not isinstance(text, str) or len(text.strip()) < 20:
        return 'unknown'
    try:
        return detect(text[:500])
    except:
        return 'unknown'

In [20]:
df_en = df[df['Language'] == 'English'].copy()
df_en['Date'] = pd.to_datetime(df_en['Date'])
df_en = df_en.dropna(subset=['text'])

In [21]:
print("Language detection... (2-3 λεπτά)")
df_en['detected_lang'] = df_en['text'].apply(detect_language)

Language detection... (2-3 λεπτά)


In [22]:
# ── Σύγκριση declared vs detected ──
print("\n" + "=" * 50)
print("DECLARED vs DETECTED LANGUAGE")
print("=" * 50)
print(f"Declared English, Detected English : "
      f"{len(df_en[df_en['detected_lang'] == 'en']):,}")
print(f"Declared English, Detected OTHER  : "
      f"{len(df_en[df_en['detected_lang'] != 'en']):,}")
print(f"\nTop detected non-English:")
print(df_en[df_en['detected_lang'] != 'en']['detected_lang'].value_counts().head(10))


DECLARED vs DETECTED LANGUAGE
Declared English, Detected English : 30,059
Declared English, Detected OTHER  : 81

Top detected non-English:
detected_lang
fr    32
de    19
id    17
it     3
nl     3
pt     2
lt     2
tl     2
es     1
Name: count, dtype: int64


In [23]:
df_en = df_en[df_en['detected_lang'] == 'en'].copy()
print(f"\nΤελικές confirmed αγγλικές ομιλίες: {len(df_en):,}")


Τελικές confirmed αγγλικές ομιλίες: 30,059
